# Star Schema Construction for Operational Risk Data

## Overview
This notebook builds a star schema data model from the processed P-COLD dataset to support efficient analytical queries and Power BI dashboarding.

The star schema organizes data into a central fact table connected to multiple dimension tables, enabling flexible slicing and aggregation.

## Objectives
- Create dimension tables for business units, risk categories, owners, and dates
- Assign surrogate keys to replace descriptive fields
- Construct a fact table at the event level
- Enable efficient joins for dashboard analysis

## Star Schema Components
- Fact Table: Operational risk events
- Dimension Tables:
  - Business Unit
  - Risk
  - Owner (placeholder)
  - Date

## Key Transformations
- Creation of date keys for time-based analysis
- Assignment of business unit and risk identifiers
- Derivation of impact levels based on loss severity

## Outputs
- `fact_risk_events_star.csv`
- `dim_business_unit.csv`
- `dim_risk_star.csv`
- `dim_owner.csv`
- `dim_date.csv`

## Notes
This schema is used directly in Power BI to support historical risk analysis and reporting.

In [1]:
# Load the processed P-COLD dimension and fact outputs created by the feature-engineering notebook.
# Dates are parsed here so they can be converted into date keys for the star schema.

import pandas as pd
import numpy as np

dim_risk = pd.read_csv("../data/processed/dim_risk.csv")
fact_events = pd.read_csv("../data/processed/fact_risk_events.csv")

fact_events["event_date"] = pd.to_datetime(fact_events["event_date"], errors="coerce")
fact_events["mitigation_due_date"] = pd.to_datetime(fact_events["mitigation_due_date"], errors="coerce")

print("dim_risk:", dim_risk.shape)
print("fact_events:", fact_events.shape)

display(dim_risk.head())
display(fact_events.head())

dim_risk: (98, 6)
fact_events: (3725, 30)


,risk_id,risk_event_type,risk_category,risk_business_line,risk_description,risk_owner
0,RISK-0001,Internal fraud,People,Commercial banking,Operational risk event: Internal fraud caused ...,Unknown
1,RISK-0002,Internal fraud,People,No business line,Operational risk event: Internal fraud caused ...,Unknown
2,RISK-0003,Internal fraud,People,Payment and settlement,Operational risk event: Internal fraud caused ...,Unknown
3,RISK-0004,"Clients, products & business practices",People,Retail banking,"Operational risk event: Clients, products & bu...",Unknown
4,RISK-0005,Internal fraud,People,Retail banking,Operational risk event: Internal fraud caused ...,Unknown


,event_id,event_start_date,event_end_date,risk_id,business_line,business_line_id,causal_factor,causal_factor_id,province_occurred,city_occurred,...,description,event_date,gross_loss,recovery_amount,net_loss,business_unit,mitigation_status,mitigation_due_date,expected_loss,probability_score
0,1,1999-08-01,2000-11-01,RISK-0001,Commercial banking,BL-003,People,CF-002,Shaanxi,Xi'an,...,Operational risk event: Internal fraud caused ...,1999-08-01,102000000.0,0.0,102000000.0,Commercial banking,Unknown,NaT,102000000.0,NaN
1,2,1998-12-01,2000-11-01,RISK-0002,No business line,BL-005,People,CF-002,Guangdong,Dongguan,...,Operational risk event: Internal fraud caused ...,1998-12-01,4200000.0,0.0,4200000.0,No business line,Unknown,NaT,4200000.0,NaN
2,3,1997-07-01,2000-04-01,RISK-0003,Payment and settlement,BL-006,People,CF-002,Guangdong,Shenzhen,...,Operational risk event: Internal fraud caused ...,1997-07-01,3320000.0,0.0,3320000.0,Payment and settlement,Unknown,NaT,3320000.0,NaN
3,4,2000-02-01,NaN,RISK-0003,Payment and settlement,BL-006,People,CF-002,Tianjin,NaN,...,Operational risk event: Internal fraud caused ...,2000-02-01,21020000.0,0.0,21020000.0,Payment and settlement,Unknown,NaT,21020000.0,NaN
4,5,1995-02-01,2000-09-01,RISK-0002,No business line,BL-005,People,CF-002,Beijing,Beijing,...,Operational risk event: Internal fraud caused ...,1995-02-01,15800000.0,0.0,15800000.0,No business line,Unknown,NaT,15800000.0,NaN


## Create Dimension Business Unit

In [2]:
# Create a business unit dimension and replace business unit names in the fact table
# with a numeric surrogate key.

dim_business_unit = (
    fact_events[["business_unit"]]
    .drop_duplicates()
    .dropna()
    .sort_values("business_unit")
    .reset_index(drop=True)
)

dim_business_unit["business_unit_id"] = np.arange(1, len(dim_business_unit) + 1)

bu_map = dict(zip(dim_business_unit["business_unit"], dim_business_unit["business_unit_id"]))
fact_events["business_unit_id"] = fact_events["business_unit"].map(bu_map)

fact_events = fact_events.drop(columns=["business_unit"])
display(dim_business_unit.head())

,business_unit,business_unit_id
0,Agency services,1
1,Asset management,2
2,Commercial banking,3
3,Corporate finance,4
4,No business line,5


## Create Dimension Owner

In [3]:
# Create a risk owner dimension.
# P-COLD does not include risk owners, so Unknown is used as a placeholder
# to preserve the star schema structure.

if "risk_owner" not in dim_risk.columns:
    dim_risk["risk_owner"] = "Unknown"

dim_owner = (
    dim_risk[["risk_owner"]]
    .drop_duplicates()
    .sort_values("risk_owner")
    .reset_index(drop=True)
)

dim_owner["owner_id"] = np.arange(1, len(dim_owner) + 1)

owner_map = dict(zip(dim_owner["risk_owner"], dim_owner["owner_id"]))
dim_risk["owner_id"] = dim_risk["risk_owner"].map(owner_map)

dim_risk = dim_risk.drop(columns=["risk_owner"])
display(dim_owner.head())

,risk_owner,owner_id
0,Unknown,1


## Create Date Dimension

In [4]:
# Create a date dimension spanning the event date range.
# Mitigation due dates are included only when available.

date_candidates = [fact_events["event_date"]]

if fact_events["mitigation_due_date"].notna().any():
    date_candidates.append(fact_events["mitigation_due_date"])

min_date = min(s.min() for s in date_candidates if s.notna().any())
max_date = max(s.max() for s in date_candidates if s.notna().any())

dim_date = pd.DataFrame({"date": pd.date_range(min_date, max_date, freq="D")})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_num"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["week_num"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_name"] = dim_date["date"].dt.strftime("%a")

display(dim_date.head())

,date,date_key,year,quarter,month_num,month_name,year_month,week_num,day_name
0,1988-06-01,19880601,1988,Q2,6,Jun,1988-06,22,Wed
1,1988-06-02,19880602,1988,Q2,6,Jun,1988-06,22,Thu
2,1988-06-03,19880603,1988,Q2,6,Jun,1988-06,22,Fri
3,1988-06-04,19880604,1988,Q2,6,Jun,1988-06,22,Sat
4,1988-06-05,19880605,1988,Q2,6,Jun,1988-06,22,Sun


## Add Date Keys

In [5]:
# Convert event and mitigation dates into integer date keys
# so the fact table can join to dim_date.

fact_events["event_date_key"] = fact_events["event_date"].dt.strftime("%Y%m%d").astype(float).astype("Int64")

fact_events["due_date_key"] = (
    fact_events["mitigation_due_date"]
    .dt.strftime("%Y%m%d")
    .astype(float)
    .astype("Int64")
)

## Create Impact Level for Star Schema

In [6]:
# Derive impact levels from net loss using severity thresholds.
# These buckets support dashboard views of loss concentration and severity.

def impact_bucket(loss):
    if pd.isna(loss):
        return "Unknown"
    elif loss < 50_000:
        return "Low"
    elif loss < 250_000:
        return "Medium"
    elif loss < 1_000_000:
        return "High"
    else:
        return "Critical"

fact_events["impact_level"] = fact_events["net_loss"].apply(impact_bucket)
display(fact_events[["net_loss", "impact_level"]].head())

,net_loss,impact_level
0,102000000.0,Critical
1,4200000.0,Critical
2,3320000.0,Critical
3,21020000.0,Critical
4,15800000.0,Critical


## Final Fact Table

In [7]:
# Select the final fact table columns at the event grain.
# This table links risk events to risk, business unit, owner, and date dimensions.

fact_risk_events = fact_events[[
    "event_id",
    "risk_id",
    "business_unit_id",
    "event_date_key",
    "due_date_key",
    "gross_loss",
    "recovery_amount",
    "net_loss",
    "impact_level",
    "probability_score",
    "expected_loss",
    "mitigation_status",
    "description"
]].copy()

display(fact_risk_events.head())

,event_id,risk_id,business_unit_id,event_date_key,due_date_key,gross_loss,recovery_amount,net_loss,impact_level,probability_score,expected_loss,mitigation_status,description
0,1,RISK-0001,3.0,19990801,<NA>,102000000.0,0.0,102000000.0,Critical,NaN,102000000.0,Unknown,Operational risk event: Internal fraud caused ...
1,2,RISK-0002,5.0,19981201,<NA>,4200000.0,0.0,4200000.0,Critical,NaN,4200000.0,Unknown,Operational risk event: Internal fraud caused ...
2,3,RISK-0003,6.0,19970701,<NA>,3320000.0,0.0,3320000.0,Critical,NaN,3320000.0,Unknown,Operational risk event: Internal fraud caused ...
3,4,RISK-0003,6.0,20000201,<NA>,21020000.0,0.0,21020000.0,Critical,NaN,21020000.0,Unknown,Operational risk event: Internal fraud caused ...
4,5,RISK-0002,5.0,19950201,<NA>,15800000.0,0.0,15800000.0,Critical,NaN,15800000.0,Unknown,Operational risk event: Internal fraud caused ...


## Export star schema tables

In [8]:
# Export the completed star schema tables for Power BI.

dim_risk.to_csv("../data/processed/dim_risk_star.csv", index=False)
dim_business_unit.to_csv("../data/processed/dim_business_unit.csv", index=False)
dim_owner.to_csv("../data/processed/dim_owner.csv", index=False)
dim_date.to_csv("../data/processed/dim_date.csv", index=False)
fact_risk_events.to_csv("../data/processed/fact_risk_events_star.csv", index=False)

print("Saved:")
print("../data/processed/dim_risk_star.csv")
print("../data/processed/dim_business_unit.csv")
print("../data/processed/dim_owner.csv")
print("../data/processed/dim_date.csv")
print("../data/processed/fact_risk_events_star.csv")

Saved:
../data/processed/dim_risk_star.csv
../data/processed/dim_business_unit.csv
../data/processed/dim_owner.csv
../data/processed/dim_date.csv
../data/processed/fact_risk_events_star.csv
